In [2]:
import os
import numpy as np
import matplotlib.pyplot as plt
import cv2
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import torch.nn as nn
import optuna
from optuna.pruners import MedianPruner
from optuna.trial import TrialState
import optuna.visualization as vis
import torch
from torch.utils.data import DataLoader
from scripts_proporciones.models import MRConvolutionalModel

# Prueba de detección de esquinas con técnicas de visión por computación

In [ ]:
# Carga de imágenes para usar tecnicas de vision con cv2 
def load_images_cv2(file):  
    img=cv2.imread(file)
    img=cv2.cvtColor(img,cv2.COLOR_BGR2RGB)

    h=img.shape[0]
    w=img.shape[1]
    if w<h: #Pasamos todas las imagenes a horizontal
        img=cv2.rotate(img,cv2.ROTATE_90_COUNTERCLOCKWISE)
    img=cv2.resize(img,(1920,1080))    
    return img

image_dir="Pruebas"
image_files=[os.path.join(image_dir, f) for f in os.listdir(image_dir) if f.endswith('.jpg')]

dataset=[load_images_cv2(i) for i in image_files]

NameError: name 'os' is not defined

In [ ]:
def four_vertices(contour):
    # Función para buscar una aproximación el contorno que tenga 4 vértices
    for i in np.linspace(0.01,0.1,50):
        aprox=cv2.approxPolyDP(contour,i*cv2.arcLength(contour,True),True)
        if len(aprox)==4:
            return aprox
    return None

In [ ]:
# Cargar imagen
img = dataset[1].copy()
gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)

# Suavizado (mejorado)
gray = cv2.equalizeHist(gray)
gray = cv2.bilateralFilter(gray, d=9, sigmaColor=75, sigmaSpace=75)

# Detección de bordes
edges = cv2.Canny(gray, 40, 120)

# Cierre morfológico para unir bordes
kernel = np.ones((5,5), np.uint8)
edges_closed = cv2.morphologyEx(edges, cv2.MORPH_CLOSE, kernel)

# Encontrar contornos externos
contours, _ = cv2.findContours(edges_closed, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
contours = [c for c in contours if cv2.contourArea(c) > 2000]

# Seleccionar contorno más grande y aplicar hull
contour = max(contours, key=cv2.contourArea)
contour = cv2.convexHull(contour)

# Aproximación adaptativa
approx = four_vertices(contour)

# Dibujar resultados
output = img.copy()
cv2.drawContours(output, [approx], -1, (0,255,0), 3)
for (x, y) in approx.reshape(-1, 2):
    cv2.circle(output, (x, y), 8, (255,0,0), -1)

plt.figure(figsize=(14,6))
plt.subplot(1,3,1); plt.imshow(gray, cmap='gray'); plt.title("Gris (bilateral)")
plt.axis("off")
plt.subplot(1,3,2); plt.imshow(edges_closed, cmap='gray'); plt.title("Bordes (cerrados)")
plt.axis("off")
plt.subplot(1,3,3); plt.imshow(output)
plt.title(f"Esquinas detectadas: {len(approx)}")
plt.axis("off")
plt.tight_layout()
plt.show()

# Mostrar coordenadas
print("Coordenadas de las esquinas detectadas:")
for i, (x, y) in enumerate(approx.reshape(-1, 2)):
    print(f"  Esquina {i+1}: (x={x}, y={y})")


# Red de clasificación y coordenadas con pytorch, búsqueda de hiperparámetros

In [ ]:
""""!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!"""
### Vamos a hacer la prueba de hiperparametros utilizando optuna ###
from scripts_recorte.corners_training import complete_training_crop
from scripts_recorte.corners_dataset import CustomImageDataset

# Elegimos la gpu si está disponible y si no la cpu
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Cargamos en data frames los conjuntos de datos ya divididos
train = pd.read_csv("./dataset_dividido/train_crop.csv")
val = pd.read_csv("./dataset_dividido/val_crop.csv")

# Creamos los datasets y dataloaders que vamos a usar para entrenar y validar
train_dataset = CustomImageDataset("./1_photos",train,True)
val_dataset = CustomImageDataset("./1_photos",val,True)


# Primero probamos pocas estapas para simplemente descartar modelos que no sean interesantes
def objective1(trial):
    try:
        # Tamaños de las capas totalmente conectadas que añadimos al modelo de base (diferentes modelos rinden mejor con diferentes tamaños)
        size1 = trial.suggest_int("size1",512,1024,step=128)
        size2 = trial.suggest_int("size2",128,512,step=64)
        # Modelo base a utilizar
        model_name = trial.suggest_categorical("model_name",['ResNet50','EfficientNet_B3','ConvNeXt_tiny','MobileNet_V3_Large'])
        # Tasa de dropout a emplear en el modelo
        dropout = trial.suggest_float("dropout",0.2,0.5)
        # Learning rates
        lr1 = trial.suggest_float("lr1", 1e-4, 1e-2, log=True)
        # Optimizador a utilizar
        optimizer = trial.suggest_categorical("optimizer",["SGD", "Adam", "AdamW", "RMSprop"])
        # Tamaño de los batches a emplear
        batch_size = trial.suggest_int("batch_size",32,64,step=32)
        
        parameters = {'size1': size1,'size2':size2,'model_name': model_name,'dropout':dropout,'lr1':lr1,'optimizer':optimizer,'batch_size':batch_size}
        print("Empezando un nuevo intento con los siguientes parámetros:",parameters)

        train_dataloader = DataLoader(train_dataset,batch_size=batch_size,shuffle=True,pin_memory=True,num_workers=2)
        val_dataloader = DataLoader(val_dataset,batch_size=64,shuffle=False,pin_memory=True,num_workers=2)

        def pruning_callback(loss, epoch):
            trial.report(loss, epoch)
            if trial.should_prune():
                print("El trial no era prometedor y va a ser podado")
                raise optuna.exceptions.TrialPruned()

        return complete_training_crop(model_name,optimizer,train_dataloader,val_dataloader,10,1,lr1,None,
                                     dropout,size1,size2,patience1=10,max_epochs1=20,fine_tuning=False,device=device,callback=pruning_callback)
    
    # Tratamiento de errores
    except optuna.exceptions.TrialPruned as e:
        raise
    except Exception as e:
        print("El trial falló, se evitarán esos hiperparámetros")
        print(e)
        raise  

# Realizamos la búsqueda de hiperparámetros utilizando optuna
pruner = MedianPruner(n_startup_trials=10,n_warmup_steps=10)
study = optuna.create_study(
    direction="minimize",
    study_name="RECORTE1",
    storage="sqlite:///Hiperparametros_recorte/Primera_prueba.db",
    load_if_exists=True,
    pruner=pruner
)
study.optimize(objective1,n_trials=20)

[I 2026-06-12 15:01:42,336] Using an existing study with name 'RECORTE1' instead of creating a new one.


Empezando un nuevo intento con los siguientes parámetros: {'size1': 512, 'size2': 128, 'model_name': 'EfficientNet_B3', 'dropout': 0.39397051650730514, 'lr1': 0.0006318026453284426, 'optimizer': 'AdamW', 'batch_size': 64}


  0%|          | 0/20 [00:05<?, ?it/s]
[W 2026-06-12 15:01:47,887] Trial 57 failed with parameters: {'size1': 512, 'size2': 128, 'model_name': 'EfficientNet_B3', 'dropout': 0.39397051650730514, 'lr1': 0.0006318026453284426, 'optimizer': 'AdamW', 'batch_size': 64} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "c:\Users\alvar\Desktop\TFG\.venv\Lib\site-packages\optuna\study\_optimize.py", line 205, in _run_trial
    value_or_values = func(trial)
  File "C:\Users\alvar\AppData\Local\Temp\ipykernel_16808\3869329038.py", line 47, in objective1
    return complete_training_crop(model_name,optimizer,train_dataloader,val_dataloader,10,1,lr1,None,
                                 dropout,size1,size2,patience1=10,max_epochs1=20,fine_tuning=False,device=device,callback=prunning_callback)
  File "c:\Users\alvar\Desktop\TFG\scripts_recorte\corners_training.py", line 281, in complete_training_crop
    obj_loss,next_epoch = train_model(model,opt_name,t

KeyboardInterrupt: 

In [13]:
# Importancia de hiperparámetros de recorte
study = optuna.load_study(
    study_name="RECORTE1",
    storage="sqlite:///Hiperparametros_recorte/Primera_prueba.db"
)

# Evaluamos la importancia de cada hiperparámetro y la representamos
fig = vis.plot_param_importances(study)
fig.show()

In [ ]:
# Mejores hiperparámetros obtenidos en la primera prueba
study = optuna.load_study(
    study_name="RECORTE1",
    storage="sqlite:///Hiperparametros_recorte/Primera_prueba.db"
)
top_trials = sorted(
    [t for t in study.trials if t.state == TrialState.COMPLETE],
    key=lambda t: t.value
)[:]
print("Hiperparámetros de los modelos ordenados del mejor al peor:")
print("-----------------------------------------------------------------------------------------------")
for i, trial in enumerate(top_trials, 1):
    print(f"Resultado: {trial.value:.4f}{"|":^3}Duración: {str(trial.duration).split(".")[0]}{"|":^3}Parámetros: {trial.params}")

Hiperparámetros de los modelos ordenados del mejor al peor:
-----------------------------------------------------------------------------------------------
Resultado: 0.0359 | Duración: 0:14:01 | Parámetros: {'size1': 896, 'size2': 384, 'model_name': 'EfficientNet_B3', 'dropout': 0.44365829717711075, 'lr1': 0.0009216753704590629, 'optimizer': 'AdamW', 'batch_size': 32}
Resultado: 0.0378 | Duración: 0:15:38 | Parámetros: {'size1': 640, 'size2': 192, 'model_name': 'EfficientNet_B3', 'dropout': 0.4301791679604592, 'lr1': 0.0012848976305700865, 'optimizer': 'AdamW', 'batch_size': 64}
Resultado: 0.0380 | Duración: 0:14:31 | Parámetros: {'size1': 512, 'size2': 192, 'model_name': 'EfficientNet_B3', 'dropout': 0.40445929046621787, 'lr1': 0.0006531761014571628, 'optimizer': 'AdamW', 'batch_size': 64}
Resultado: 0.0380 | Duración: 0:15:32 | Parámetros: {'size1': 512, 'size2': 192, 'model_name': 'EfficientNet_B3', 'dropout': 0.4161570806900197, 'lr1': 0.0010977308320440611, 'optimizer': 'AdamW', 

In [ ]:
"""
Si observamos primero las importancias de los hiperparámetros y luego observarmos cuales son los mejores obtenidos, concluimos que el mejor optimizador en este caso es el
AdamW y tiene mucha importancia por lo que será el que elijamos. En cuanto al learning rate que también es importante, parece que el 1e-3 o valores un poco menores suelen
funcionar mejor, aunque es cierto que en esta prueba teníamos pocas etapas por lo que se suele tender a premiar lr más altos, pero aun asi no se han obtenido muchos
resultados con valores cercanos al lr mas alto permitido que era 1e-2 por lo que el lr no es bueno solo por ser grande y tenemos un scheduler implementado que se encarga
de reducir el lr si pasan varias epocas sin mejorar, por todo esto nos quedaremos inicialmente con ese lr de 1e-3 que disminuirá si el modelo se va estancando.
A la hora de elegir el modelo parece que con el número limitado de imágenes para entrenar que tenemos y la naturaleza de este problema, EfficientNet_B3 es el modelo base
que mejor funciona y es el que vamos a entrenar.
otros hiperparámetros como el batch size o los tamaños de las capas parecen no tener demasiada importancia, e incluso el dropout no aparece ni siquiera representado.
"""

# Rendimiento en conjunto de test (Tarea de recorte)

In [2]:
from scripts_recorte.corners_model import TransferLearning
from sklearn.metrics import precision_score, recall_score
from scripts_recorte.corners_dataset import CustomImageDataset

torch.manual_seed(67)
torch.cuda.manual_seed_all(67)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

loss_module_coords = nn.MSELoss()
mae_loss = nn.L1Loss()
loss_module_corners = nn.BCEWithLogitsLoss()

test_mae=0
test_bce=0
test_mse=0
correct_predictions=0
total_predictions=0
total_positives=0
corner_trues=[]
corner_preds=[]

test = pd.read_csv("./dataset_dividido/test_crop.csv")
test_dataset = CustomImageDataset("./1_photos",test,True)
test_dataloader = DataLoader(test_dataset,batch_size=32,shuffle=False,pin_memory=True)

model = TransferLearning("EfficientNet_B3",0.4,896,384).to(device)
model.load_state_dict(torch.load(f"./Modelos/{model.name}_model.pth",map_location=device))

with torch.no_grad():
    model.eval()
    for data_inputs,data_labels in test_dataloader:
        corner_trues.extend(data_labels["classification"].numpy())
        data_inputs = data_inputs.to(device)
        data_corners_labels = data_labels["classification"].to(device)
        data_coords_labels = data_labels["coordinates"].to(device)

        logits = model(data_inputs)
        corners_logits = logits["classification"].squeeze(1)
        coords_logits = logits["coordinates"]

        # Con esto calcularemos la accuracy para el conjunto de validacion
        predicted_classes = (corners_logits > 0).int()
        corner_preds.extend(predicted_classes.detach().cpu().numpy())
        correct_predictions += (predicted_classes == data_corners_labels).sum().item()
        total_predictions += data_corners_labels.size(0)

        # Solo nos interesan las coordenadas si tiene esquinas
        has_corners_mask = data_corners_labels.bool()
        positives_in_batch = has_corners_mask.sum().item()
        total_positives += positives_in_batch

        # Si hay alguna imagen con esquinas
        if positives_in_batch>0:
            # Filtrar predicciones y targets
            reg_pred_corners = coords_logits[has_corners_mask]
            reg_true_corners = data_coords_labels[has_corners_mask]

            # Calcular la pérdida y MAE SOLO en los ejemplos positivos
            # Primero multiplicamos las losses por el número de datos con esquinas para luego que sea proporcional al número de imagenes positivas y no al número de batches
            test_mse += loss_module_coords(reg_pred_corners, reg_true_corners).item()*positives_in_batch
            test_mae += mae_loss(reg_pred_corners, reg_true_corners).item()*positives_in_batch
        test_bce += loss_module_corners(corners_logits, data_corners_labels).item()

    # La clase positiva va a ser "Tiene esquinas"
    precision = precision_score(corner_trues,corner_preds,pos_label=1,zero_division=0)
    recall = recall_score(corner_trues,corner_preds,pos_label=1,zero_division=0)
    accuracy = correct_predictions / total_predictions

    test_bce_loss = test_bce/len(test_dataloader)
    test_mse_loss = test_mse/max(1,total_positives)
    test_mae_loss = test_mae/max(1,total_positives)
print(f"Test Corners BCE {test_bce_loss:.4f}, Test MSE {test_mse_loss:.4f}, Test MAE {test_mae_loss:.4f}\nAccuracy {accuracy*100:.2f}%, Precision {precision*100:.2f}%, Recall {recall*100:.2f}%")

"""
En vista de los resultados podemos ver que son peores que los de validación, en el caso de las coordenadas el MAE aumenta en 0.01 aproximadamente que es bastante,
pero 0.0314 sigue siento un valor decente para la tarea. En el caso de la clasificación, vemos un BCE de 0.1669 que es mucho peor que el de validación, lo que parece
indicar un fuerte overfitting que ha pasado desapercibido por la baja cantidad y variedad de ejemplos, aún asi los valores de accuracy, precision y recall son bastante
buenos, ya que el recall del 100% nos asegura que no se pierden imágenes de test que tengan un cuadrado con esquinas. Teniendo en cuenta que la mayoría de ejemplos son
de la clase positiva y que tenemos bastantes pocos para entrenar un modelo de este tipo (unas 500 imagenes) es normal que haya limitaciones en el entrenamiento, este 
modelo quedaría planteado para una posible revisión y un reentrenamiento con más cantidad de ejemplos y una posible validación cruzada con el objetivo de optimizarlo.
(Cambiar los pesos de coordenadas o de la detección de esquinas no tenía un efecto positivo en este problema)
"""

Test Corners BCE 0.1752, Test MSE 0.0010, Test MAE 0.0174
Accuracy 98.88%, Precision 98.82%, Recall 100.00%


'\nEn vista de los resultados podemos ver que son peores que los de validación, en el caso de las coordenadas el MAE aumenta en 0.01 aproximadamente que es bastante,\npero 0.0314 sigue siento un valor decente para la tarea. En el caso de la clasificación, vemos un BCE de 0.1669 que es mucho peor que el de validación, lo que parece\nindicar un fuerte overfitting que ha pasado desapercibido por la baja cantidad y variedad de ejemplos, aún asi los valores de accuracy, precision y recall son bastante\nbuenos, ya que el recall del 100% nos asegura que no se pierden imágenes de test que tengan un cuadrado con esquinas. Teniendo en cuenta que la mayoría de ejemplos son\nde la clase positiva y que tenemos bastantes pocos para entrenar un modelo de este tipo (unas 500 imagenes) es normal que haya limitaciones en el entrenamiento, este \nmodelo quedaría planteado para una posible revisión y un reentrenamiento con más cantidad de ejemplos y una posible validación cruzada con el objetivo de optimi

# Preprocesado de las etiquetas

In [9]:
# Cargamos el excel usando pandas y nos quedamos solo con las columnas de las etiquetas y la del nombre de la foto
# Además vamos a quitar la palabra cobertura de las columnas de las etiquetas y poner todos los índices en minúscula para una mayor legibilidad.

cols = list(range(27,39))
labels_route = "2_labels/1_GROUNDTRUTH_table_NORGRASS25_clean.xlsx"
df = pd.read_excel(labels_route,usecols=cols)
df.columns = df.columns.str.lower().str.replace("cobertura","").str.strip()
target_cols = ['n. noltei','z. marina','g. vermiculophylla','sedimento','arena','roca','algas verdes','algas pardas','algas rojas','microfitobentos']

# Nos quedamos solo con los registros que tengan alguna de las fotos recortadas en su campo "foto"
# Esto nos ahorra además tener que eliminar los registros que tengan un valor vacío en la columna correspondiente
cropped_photos = [n for n in os.listdir("fotos_recortadas") if n.endswith((".jpg",".jpeg",".png"))]
df = df[df["foto"].isin(cropped_photos)]

# Vamos a pasar las columnas a float para evitar futuros errores
df[target_cols] = df[target_cols].astype(float)

# La columna "punto verificación" tiene muchos valores faltantes, pero solo la hemos cargado para un estudio posterior por lo que vamos 
#   a rellenar estos valores faltantes con 0s sin detenernos mucho
df["punto verificacion"] = df["punto verificacion"].fillna(0)
df


,punto verificacion,n. noltei,z. marina,g. vermiculophylla,sedimento,arena,roca,algas verdes,algas pardas,algas rojas,microfitobentos,foto
1,35.0,0.0,0.0,0.0,0.0,100.0,0.0,0.0,0.0,0.0,0.0,PHOTO_20250521_191501.jpg
2,35.0,0.0,0.0,0.0,0.0,100.0,0.0,0.0,0.0,0.0,0.0,PHOTO_20250521_191354.jpg
3,35.0,0.0,0.0,0.0,0.0,100.0,0.0,0.0,0.0,0.0,0.0,PHOTO_20250521_191329.jpg
4,35.0,0.0,0.0,0.0,0.0,100.0,0.0,0.0,0.0,0.0,0.0,PHOTO_20250521_191305.jpg
5,35.0,0.0,0.0,0.0,0.0,100.0,0.0,0.0,0.0,0.0,0.0,PHOTO_20250521_191241.jpg
...,...,...,...,...,...,...,...,...,...,...,...,...
2446,0.0,50.0,0.0,0.0,20.0,0.0,0.0,30.0,0.0,0.0,0.0,PHOTO_20250728_114615.jpg
2447,0.0,40.0,0.0,0.0,50.0,0.0,0.0,10.0,0.0,0.0,0.0,PHOTO_20250728_114413.jpg
2448,0.0,0.0,0.0,0.0,92.0,0.0,0.0,5.0,3.0,0.0,0.0,PHOTO_20250728_113013.jpg
2449,0.0,50.0,0.0,0.0,30.0,0.0,0.0,20.0,0.0,0.0,0.0,PHOTO_20250728_113931.jpg


## Tratamiento de registros duplicados en función del campo foto

In [10]:
# -- Vamos a estudiar las carácterísticas de los registros que usan por algún motivo la misma foto
dups=df[df.duplicated(subset=["foto"],keep=False)]

# False - coinciden los valores de todos los campos para los registros con la misma foto (básicamente registros idénticos duplicados)
# True  - no coinciden los valores de todos los campos para los registros con la misma foto (un registro es el verdadero y los demás tienen mediciones incorrectas)
complete_dups=(dups.groupby("foto").nunique()[target_cols]!=1).any(axis=1).reset_index(name="dups")
incorrect_dups=complete_dups[complete_dups["dups"]]["foto"]
dups=dups[dups["foto"].isin(incorrect_dups)]
# De momento solo nos preocupamos por aquellos duplicados con diferente porcentaje de cada clase porque de los otros nos basta con quedarnos con uno de ellos

# -- Ahora no nos queda más remedio que ir comprobando para cada conjunto de duplicados la foto original y decidir cúal de los registros se corresponde con ella
# Podríamos eliminar directamente este tipo de duplicados del conjunto de datos y quedarnos solo con los duplicados que coinciden entre sí limpios
#  pero como son pocos vamos a elegir uno del otro tipo de duplicados que se corresponda con la imagen
# (vamos a ignorar de momento que a veces no suman 100 las columnas)
dups


,punto verificacion,n. noltei,z. marina,g. vermiculophylla,sedimento,arena,roca,algas verdes,algas pardas,algas rojas,microfitobentos,foto
182,0.0,95.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,10.0,0.0,PHOTO_20250521_170652.jpg
183,46.0,0.0,0.0,0.0,20.0,80.0,0.0,0.0,0.0,0.0,0.0,PHOTO_20250521_170652.jpg
261,0.0,0.0,0.0,0.0,0.0,0.0,0.0,100.0,0.0,0.0,0.0,PHOTO_20250523_205348.jpg
262,0.0,0.0,80.0,0.0,0.0,0.0,0.0,20.0,0.0,0.0,0.0,PHOTO_20250523_205348.jpg
357,0.0,80.0,0.0,0.0,20.0,0.0,0.0,0.0,0.0,0.0,0.0,PHOTO_20250523_194738.jpg
358,0.0,50.0,0.0,0.0,50.0,0.0,0.0,0.0,0.0,0.0,0.0,PHOTO_20250523_194738.jpg
389,0.0,0.0,0.0,0.0,100.0,0.0,0.0,0.0,0.0,0.0,0.0,PHOTO_20250523_191013.jpg
390,0.0,100.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,PHOTO_20250523_191013.jpg
414,0.0,0.0,0.0,0.0,60.0,0.0,0.0,40.0,0.0,0.0,0.0,PHOTO_20250523_184406.jpg
415,0.0,100.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,PHOTO_20250523_184406.jpg


In [11]:
""""
Después de analizar las fotos más fácilmente distinguibles llegamos a la conclusión de que el último registro es siempre el correcto en estos casos, por tanto
como tenemos pocas imagenes restantes que sean problemáticas podríamos suponer que en todos los casos la correcta es el segundo registro, esto significaría
seguramente que se cargaron dos imágenes con el mismo nombre pero la segunda reemplazó a la primera y por eso es el último registro el correcto.
Aún asi asumiremos que no siempre se cumplirá y vamos primero a eliminar todos los registros de fotos problemáticas que sea difícil distinguir el correcto, y
luego de los que queden nos quedaremos con el último registro, en este caso se pierden solo 5 y así evitamos datos mal etiquetados en la medida de lo posible
"""
# Lista de los duplicados problemáticos que vamos a eliminar
delete_index=[357,358,821,822,1104,1105,1247,1248,1875,1876]
df=df.drop(index=delete_index)
df=df.drop_duplicates(subset="foto",keep="last")

## Tratamiento de valores faltantes o incoherentes en las columnas objetivo

In [12]:
# Primero hacemos una comprobación rápida para ver si hay valores faltantes en nuestras columnas objetivo
print(df[df[target_cols].isna().any(axis=1)])
# Nos devuelve un empty data frame porque no hay valores faltantes

# Sabemos que nuestras columnas objetivo incluyen el porcentaje de cada clase que hay presente en la imagen
# Por tanto la suma de las columnas para un registro siempre debería sumar 100, vamos a comprobar si hay 
#  registros en los que la suma de más o menos de 100 y estudiaremos el posible por qué y buscaremos una solución
wrong_values =df[df.loc[:,target_cols].sum(axis=1)!=100]
wrong_values

Empty DataFrame
Columns: [punto verificacion, n. noltei, z. marina, g. vermiculophylla, sedimento, arena, roca, algas verdes, algas pardas, algas rojas, microfitobentos, foto]
Index: []


,punto verificacion,n. noltei,z. marina,g. vermiculophylla,sedimento,arena,roca,algas verdes,algas pardas,algas rojas,microfitobentos,foto
6,0.0,0.0,0.0,0.0,20.0,0.0,0.0,0.0,0.0,0.0,70.0,PHOTO_20250521_190410.jpg
67,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,30.0,0.0,0.0,PHOTO_20250521_182315.jpg
118,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,95.0,0.0,0.0,PHOTO_20250521_173734.jpg
119,0.0,0.0,0.0,0.0,0.0,0.0,0.0,95.0,0.0,0.0,0.0,PHOTO_20250521_173603.jpg
179,46.0,0.0,0.0,0.0,10.0,0.0,0.0,0.0,0.0,0.0,0.0,PHOTO_20250521_170823.jpg
...,...,...,...,...,...,...,...,...,...,...,...,...
2393,0.0,30.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,PHOTO_20250728_135838.jpg
2395,0.0,5.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,PHOTO_20250728_135552.jpg
2406,0.0,85.0,0.0,0.0,0.0,0.0,0.0,25.0,0.0,0.0,0.0,PHOTO_20250728_132836.jpg
2439,1.0,0.0,0.0,0.0,90.0,0.0,0.0,5.0,0.0,0.0,0.0,PHOTO_20250728_115900.jpg


In [13]:
"""
La primera comprobación que hacemos es mirar la columna "punto verificación", esta columna no tiene nada que ver con las columnas objetivo que tienen los porcentajes,
sin embargo es una columna numérica que está justo delante de estas columnas objetivo y por tanto es posible que por algún desplazamiento u otro problema los valores
que deberían estar en nuestros atributos se hayan quedado en la columna incorrecta.
De los registros en los que la suma no daba el valor 100, hay 6 registros que si lo sumarían si contamos el campo punto verificación, vamos a revisar si tendría sentido
desplazar por ejemplo todas sus mediciones una posición a la derecha.
Tras analizarlos, vemos que los 5 primeros registros si parecen estar desplazados por lo que vamos a sustituir sus valores por los que parecen ser correctos, sin embargo,
el último registro parece no encajar y dentro de no sumar 100 correctamente si se pueden ver los atributos en una proporción más o menos correspondiente a la que se indica.
"""
numeric_cols=["punto verificacion"]+target_cols
wrong_values[wrong_values.loc[:,numeric_cols].sum(axis=1)==100]


,punto verificacion,n. noltei,z. marina,g. vermiculophylla,sedimento,arena,roca,algas verdes,algas pardas,algas rojas,microfitobentos,foto
375,90.0,0.0,0.0,0.0,10.0,0.0,0.0,0.0,0.0,0.0,0.0,PHOTO_20250523_192808.jpg
746,90.0,0.0,0.0,0.0,10.0,0.0,0.0,0.0,0.0,0.0,0.0,PHOTO_20250522_165855.jpg
1049,80.0,0.0,0.0,0.0,20.0,0.0,0.0,0.0,0.0,0.0,0.0,PHOTO_20250524_210346.jpg
1053,40.0,0.0,0.0,0.0,60.0,0.0,0.0,0.0,0.0,0.0,0.0,PHOTO_20250524_205938.jpg
1089,80.0,0.0,0.0,0.0,20.0,0.0,0.0,0.0,0.0,0.0,0.0,PHOTO_20250524_205003.jpg
1451,15.0,0.0,0.0,80.0,5.0,0.0,0.0,0.0,0.0,0.0,0.0,PHOTO_20250525_110835.jpg


In [ ]:
# Los 5 registros desplazados los sustituímos por sus valores correctos y volvemos a comprobar que registros suman un valor diferente de 100
shifted_rows=wrong_values[wrong_values.loc[:,numeric_cols].sum(axis=1)==100][:-1]
df.loc[shifted_rows.index,target_cols]=shifted_rows.iloc[:,0:10].values

# Al arreglar esas filas y volver a quedarnos con las incorrectas vemos que hay 5 menos
wrong_values=df[df.loc[:,target_cols].sum(axis=1)!=100][target_cols+["foto"]]
wrong_values

,n. noltei,z. marina,g. vermiculophylla,sedimento,arena,roca,algas verdes,algas pardas,algas rojas,microfitobentos,foto
6,0.0,0.0,0.0,20.0,0.0,0.0,0.0,0.0,0.0,70.0,PHOTO_20250521_190410.jpg
67,0.0,0.0,0.0,0.0,0.0,0.0,0.0,30.0,0.0,0.0,PHOTO_20250521_182315.jpg
118,0.0,0.0,0.0,0.0,0.0,0.0,0.0,95.0,0.0,0.0,PHOTO_20250521_173734.jpg
119,0.0,0.0,0.0,0.0,0.0,0.0,95.0,0.0,0.0,0.0,PHOTO_20250521_173603.jpg
179,0.0,0.0,0.0,10.0,0.0,0.0,0.0,0.0,0.0,0.0,PHOTO_20250521_170823.jpg
...,...,...,...,...,...,...,...,...,...,...,...
2393,30.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,PHOTO_20250728_135838.jpg
2395,5.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,PHOTO_20250728_135552.jpg
2406,85.0,0.0,0.0,0.0,0.0,0.0,25.0,0.0,0.0,0.0,PHOTO_20250728_132836.jpg
2439,0.0,0.0,0.0,90.0,0.0,0.0,5.0,0.0,0.0,0.0,PHOTO_20250728_115900.jpg


In [8]:
"""
Lo que haremos ahora es comprobar para los demás ejemplos mal etiquetados, comprobar si al menos los elementos que las etiquetas
están indicando presentes en las fotos realmente lo están y veremos si la proporción que proponen tiene sentido, es decir,
si pone que hay 80 arena y 10 alga verde, pero después la foto está llena de algas verdes y casi no hay arena pués no tendrá
sentido, por el contrario si pone 40 arena y 40 alga verde, si después en la imagen hay mitad arena mitad alga verde, es posible
que los valores reales sean 50|50 y la proporción relativa sea correcta entre ellos, otro caso que podría darse es que ponga
40 arena y 40 alga verde, pero luego en la imagen haya algo de alga roja que faltaba por identificarse.
Si lo que se da es el primer o tercer caso, lo más probable es que esos registros sean inservibles y tengamos que eliminarlos.
"""
"""
Después de una primera inspección por encima podemos sacar una conclusion, que en los casos donde solo hay un elemento visible
según la etiqueta parece que faltan otros elementos por mencionar, en la imagen se ven claramente varias cosas diferentes pero en la etiqueta
solo se recoge una de ellas con su proporción, es decir que las etiquetas están incompletas puesto que les faltan elementos que no consideran.
Por tanto, vamos a empezar por el caso de que solo se indique un elemento en la imagen.
"""
one_target=wrong_values[((wrong_values.loc[:,target_cols]>0).sum(axis=1))==1]
one_target

,n. noltei,z. marina,g. vermiculophylla,sedimento,arena,roca,algas verdes,algas pardas,algas rojas,microfitobentos,foto
67,0.0,0.0,0.0,0.0,0.0,0.0,0.0,30.0,0.0,0.0,PHOTO_20250521_182315.jpg
118,0.0,0.0,0.0,0.0,0.0,0.0,0.0,95.0,0.0,0.0,PHOTO_20250521_173734.jpg
119,0.0,0.0,0.0,0.0,0.0,0.0,95.0,0.0,0.0,0.0,PHOTO_20250521_173603.jpg
179,0.0,0.0,0.0,10.0,0.0,0.0,0.0,0.0,0.0,0.0,PHOTO_20250521_170823.jpg
235,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,35.0,PHOTO_20250521_163601.jpg
387,0.0,0.0,0.0,0.0,0.0,0.0,5.0,0.0,0.0,0.0,PHOTO_20250523_191635.jpg
440,90.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,PHOTO_20250523_182443.jpg
480,90.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,PHOTO_20250523_180733.jpg
1411,95.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,PHOTO_20250525_122037.jpg
1850,90.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,PHOTO_20250527_124525.jpg


In [9]:
"""
La revisión de estos ejemplos es muy rápida porque tendrían que ser fotos uniformes con un único elemento presente, sin embargo,
al revisar las fotografías podemos confirmar la hipótesis inicial de que las etiquetas están incompletas puesto que a simple
vista somos capaces de identificar la presencia de múltiples otros elementos que no indican.
De esta manera vamos a descartar todos los registros en los que se dé esta situación de nuestro conjunto de datos y después
revisaremos los ejemplos con porcentajes incorrectos pero varios elementos recogidos.
"""
df=df.drop(index=one_target.index)

In [10]:
mul_target=wrong_values[((wrong_values.loc[:,target_cols]>0).sum(axis=1))>1].copy()
mul_target

,n. noltei,z. marina,g. vermiculophylla,sedimento,arena,roca,algas verdes,algas pardas,algas rojas,microfitobentos,foto
6,0.0,0.0,0.0,20.0,0.0,0.0,0.0,0.0,0.0,70.0,PHOTO_20250521_190410.jpg
240,70.0,0.0,0.0,45.0,0.0,0.0,5.0,0.0,0.0,0.0,PHOTO_20250521_162844.jpg
352,40.0,0.0,0.0,20.0,0.0,0.0,20.0,0.0,0.0,0.0,PHOTO_20250523_194929.jpg
722,60.0,0.0,0.0,25.0,0.0,0.0,5.0,0.0,0.0,0.0,PHOTO_20250522_170603.jpg
758,85.0,0.0,0.0,25.0,0.0,0.0,0.0,0.0,0.0,0.0,PHOTO_20250522_165529.jpg
777,65.0,0.0,0.0,25.0,0.0,0.0,0.0,0.0,0.0,0.0,PHOTO_20250522_164804.jpg
871,0.0,0.0,0.0,5.0,0.0,0.0,0.0,90.0,0.0,0.0,PHOTO_20250522_084637.jpg
874,0.0,0.0,0.0,10.0,0.0,0.0,0.0,80.0,0.0,0.0,PHOTO_20250522_084432.jpg
897,0.0,0.0,0.0,0.0,0.0,50.0,20.0,25.0,0.0,0.0,PHOTO_20250522_083254.jpg
1066,20.0,0.0,75.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,PHOTO_20250524_205447.jpg


In [11]:
""""
Si ahora comprobamos las imágenes de los registros que parecen tener porcentajes incompletos vemos que la mayoría de ellos
tienen sentido, los elementos que aparecen en las etiquetas están presentes en las fotografías y la proporción de estos tiene
sentido, aunque no llegue al 100% parece que si reducimos sus porcentajes proporcionalmente si se llegará a ese valor deseado,
pero hay algún ejemplo específico que tiene errores evidentes y tenemos que arreglar de forma específica:
    - La fila 2368 tiene un claro error en la columna "algas rojas" en la que tiene el valor 1111 que si eliminamos
      nos permite sumar 100 entre las columnas.

    - Hay una serie de filas que tienen valores decimales menores que 1 que parecen haber sido introducidos incorrectamente
      puesto que si movemos la coma a la derecha y en lugar de un 0,5 nos quedamos con un 5 la suma de las columnas en
      total nos daría el valor 100 que buscamos, habría dos excepciones, la fila 2271 en la que tenemos, a parte de un
      valor 0.5, un valor 70.5 que dabería ser 75 para que la suma sea correcta; la otra excepción es la fila 2242 en la que
      el valor 0.5 no debería ser un 5 sino que simplemente sobra y debería ser un 0.
      
El resto de casos deberíamos ajustarlos proporcionalmente para que la relación de valores entre columnas sea la misma pero sumen
la cantidad esperada que es 100
"""
# Cambiamos los valores incorrectos que deberían ser 0
mul_target.loc[2368,"algas pardas"] = 0
mul_target.loc[2242,"algas rojas"] = 0

# Empezamos quedándonos solo con la parte decimal y la multiplicamos por 10 y luego se la sumamos a la parte entera de cada posición
mul_target.loc[:,target_cols] = mul_target[target_cols]//1+(mul_target[target_cols]%1)*10


# Vamos a ajustar proporcionalmente las columnas, para ello primero dividimos entre la suma de cada fila y multiplicamos por 100
values = round(mul_target[target_cols].div(mul_target[target_cols].sum(axis=1),axis=0)*100,1)
# Le hemos permitido un decimal a las columnas pero queremos asegurarnos de que sumen exactamente 100 y no haya problemas de suma de 
# decimales que pueda quedar por ejemplo en un 99.9 o en un 100.1 por problemas de precisión
# Para evitar eso vamos a comprobar el residuo que falta o sobra para llegar a 100 de cada fila y se lo sumamos a la columna con el
# valor más grande puesto que será en la que menos se note la variación
residual = 100 - values.sum(axis=1)
max_col = values.idxmax(axis=1)
for i in values.index:
    values.loc[i,max_col.loc[i]] += residual.loc[i]
df.loc[mul_target.index,target_cols] = values

# Comprobamos ahora si queda algún valor incoherente
df[df.loc[:,target_cols].sum(axis=1)!=100][target_cols+["foto"]]
# Es muy importante para nuestros modelos que las clases sumen exactamente 1 al convertirlas por lo que tendrán que sumar 100 o 1

,n. noltei,z. marina,g. vermiculophylla,sedimento,arena,roca,algas verdes,algas pardas,algas rojas,microfitobentos,foto


In [ ]:
# Como podemos ver no queda ningún registro que tenga incoherencias en las sumas y con esto concluimos el preprocesado.
# Ahora lo que hacemos es guardar el data frame limpio con las columnas que nos interesarán

# df[target_cols+["foto"]].to_csv("etiquetas_fotos.csv",index=False)

In [13]:
df.columns

Index(['punto verificacion', 'n. noltei', 'z. marina', 'g. vermiculophylla',
       'sedimento', 'arena', 'roca', 'algas verdes', 'algas pardas',
       'algas rojas', 'microfitobentos', 'foto'],
      dtype='object')

In [12]:
"""
Vamos a estudiar la presencia de cada clase en las fotografías para comprobar si las clases están desbalanceadas y nos puede convenir darles un tratamiento diferente.
Como podemos ver que hay varias clases que aparecen en una minoría de imagenes nos interesa tener cuidado con esos elementos puesto que nuestro modelo puede aprender
a predecir siempre cero para ellos y en la mayoría de imágenes sería correcto y por tanto no sería útil a la hora de predecir dichas clases.
Podría ser buena idea aplicar data augmentation a las imágenes pero al tener tan pocas fotos con presencia de estas clases comparado con las imágenes totales al final
o la variedad no será muy buena y tendremos muchas fotos "repetidas" con pequeños cambios, o si usamos menos fotos originales estamos desaprovechando etiquetas que
tenemos disponibles
"""
# Comprobamos el número de fotos en las que aparece cada clase
print("Presencia en fotos")
print((df[target_cols]>0).sum())

# Comprobamos la proporción de fotos en las que aparece cada imagen
print("\nPorcentaje de presencia en fotos")
print(round((df[target_cols]>0).sum()/len(df)*100,2))

# Comprobar que peso equipararía las clases para el entrenamiento

Presencia en fotos
n. noltei             1256
z. marina               44
g. vermiculophylla     118
sedimento             1139
arena                  367
roca                    55
algas verdes           616
algas pardas           150
algas rojas             23
microfitobentos         32
dtype: int64

Porcentaje de presencia en fotos
n. noltei             59.50
z. marina              2.08
g. vermiculophylla     5.59
sedimento             53.96
arena                 17.39
roca                   2.61
algas verdes          29.18
algas pardas           7.11
algas rojas            1.09
microfitobentos        1.52
dtype: float64


In [ ]:
# Vamos a estudiar las porporciones generales en las imágenes para las diferentes clases
fig = go.Figure()

# Agrupamos las clases en una misma columna
df_long = df[target_cols].melt(var_name="clase",value_name="proporcion")

fig = px.box(
    df_long,
    x="clase",
    y="proporcion",
    color="clase",
    points="outliers",
    title="Distribución de proporciones por clase",
)

fig.update_layout(
    yaxis=dict(title="Proporción por foto",range=[0, 100]),
    xaxis=dict(title="Clases"),
    height=550,
    showlegend=False,
)

# Visualizamos el diagrama de cajas general
fig.show()

In [23]:
# Ahora vamos a explorar las proporciones de las clases solo para las imágenes en las que estan presentes
fig = go.Figure()

# Agrupamos las clases en una misma columna y nos quedamos con los casos donde su proporción sea mayor que 0
df_long = df[target_cols].melt(var_name="clase",value_name="proporcion")
df_long = df_long[df_long["proporcion"]>0]

fig = px.box(
    df_long,
    x="clase",
    y="proporcion",
    color="clase",
    points="outliers",
    title="Distribución de proporciones por clase",
)

fig.update_layout(
    yaxis=dict(title="Proporción por foto",range=[0,100]),
    xaxis=dict(title="Clases"),
    height=550,
    showlegend=False,
    font=dict(size=20)
)

# Dibujamos el diagrama de cajas considerando para cada clase las imágenes en las que aparecen
fig.show()

# Modelos de estimación de proporciones, búsqueda de hiperparámetros

## Primer modelo - Base convolucional

In [ ]:
""""!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!"""
### Vamos a hacer la prueba de hiperparametros utilizando optuna ###
from scripts_proporciones.models_training import complete_training
from scripts_proporciones.create_dataset import CustomImageDataset

# Elegimos la gpu si está disponible y si no la cpu
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Cargamos en data frames los conjuntos de datos ya divididos
train = pd.read_csv("./dataset_dividido/train.csv")
val = pd.read_csv("./dataset_dividido/val.csv")

# Creamos los datasets y dataloaders que vamos a usar para entrenar y validar
train_dataset = CustomImageDataset("./fotos_recortadas",train,True)
val_dataset = CustomImageDataset("./fotos_recortadas",val,True)

# Primeros valores de hiperparámetros a probar
def objective_MRConvolutional_base(trial):
    try:
        # Modelo base a utilizar
        model_name = trial.suggest_categorical("model_name",["RegNet_Y_3_2GF","ConvNeXt_tiny","ResNet50","EfficientNetV2_small"])
        # Optimizador a utilizar
        optimizer = trial.suggest_categorical("optimizer",["SGD","Adam","AdamW","RMSprop"])
        # Lr a utilizar (exploramos solo unos pocos posibles para agilizar el proceso de selección de base)
        lr = trial.suggest_categorical("learning rate",[1e-2,1e-3,1e-4,1e-5])
        # Tamaños de las capas totalmente conectadas que añadimos al modelo de base (diferentes modelos rinden mejor con diferentes tamaños)
        size1 = trial.suggest_int("size1",512,1024,step=128)
        size2 = trial.suggest_int("size2",128,512,step=64)

        # Juntamos los hiperparámetros a optimizar"
        parameters = {"model_name":model_name,"optimizer":optimizer,"learning rate":lr,"size1":size1,"size2":size2}

        print("Empezando un nuevo intento con los siguientes parámetros:",parameters)
        # Esta función se llama al final de cada época y sirve para podar las ejecuciones menos prometedoras
        def pruning_callback(loss, epoch):
            trial.report(loss, epoch)
            if trial.should_prune():
                print("El trial no era prometedor y va a ser podado")
                raise optuna.exceptions.TrialPruned()

        train_dataloader = DataLoader(train_dataset, batch_size=64,shuffle=True,pin_memory=True,num_workers=2)
        val_dataloader = DataLoader(val_dataset, batch_size=64,shuffle=False,pin_memory=True,num_workers=2)

        return complete_training("MRConvolutional",model_name,optimizer,train_dataloader,val_dataloader,lr1=lr,lr2=None,dropout=0.4,fine_tuning=False,
                                 size1=size1,size2=size2,patience1=10,patience2=None,max_epochs1=20,max_epochs2=None,label_smoothing=0.01,device=device,callback=pruning_callback)
    
    # Tratamiento de errores
    except optuna.exceptions.TrialPruned as e:
        raise
    except Exception as e:
        print("El trial falló, se evitarán esos hiperparámetros")
        print(e)
        raise  

# Realizamos la búsqueda de hiperparámetros utilizando optuna y podando las combinaciones que no sean prometedoras
pruner = MedianPruner(n_startup_trials=15,n_warmup_steps=10)
study = optuna.create_study(
    direction="minimize",
    study_name="TFG_elegir_base_MRConvolutionalModel",
    storage="sqlite:///Hiperparametros_MR/Primera_prueba_MRConvolutional.db",
    load_if_exists=True,
    pruner=pruner,
)
study.optimize(objective_MRConvolutional_base,n_trials=50)

KeyboardInterrupt: 

In [3]:
# Importancia de hiperparámetros de proporciones
study = optuna.load_study(
    study_name="TFG_elegir_base_MRConvolutionalModel",
    storage="sqlite:///Hiperparametros_MR/Primera_prueba_MRConvolutional.db"
)

# Evaluamos la importancia de cada hiperparámetro y la representamos
fig = vis.plot_param_importances(study)
fig.show()

In [ ]:
# Estudiamos los hiperparámetros con mejores resultados de la primera prueba y asi elegimos el mejor modelo base

# Mejores parámetros obtenidos en la primera prueba
study = optuna.load_study(
    study_name="TFG_elegir_base_MRConvolutionalModel",
    storage="sqlite:///Hiperparametros_MR/Primera_prueba_MRConvolutional.db"
)
top_trials = sorted(
    [t for t in study.trials if t.state == TrialState.COMPLETE],
    key=lambda t: t.value
)[:20]
print("Hiperparámetros de los modelos ordenados del mejor al peor:")
print("-----------------------------------------------------------------------------------------------")
for i, trial in enumerate(top_trials, 1):
    print(f"Resultado: {trial.value:.4f}{"|":^3}Duración: {str(trial.duration).split(".")[0]}{"|":^3}Parámetros: {trial.params}")

""""
Vemos claramente que la mejor base convolucional es ConvNeXt_tiny con los optimizadores Adam y AdamW entre los que podemos todavía hacer una comparativa más
profunda para elegir el mejor de los dos, el mejor learning rate con este modelo específico parece estar en torno al 0.001, en cuanto al tamaño de las capas
totalmente conectadas que tenemos añadirdas parece que los valores están en  torno al 1000 para la primera y el 300-400.
Para el siguiente ajuste de hiperparámetros intentaremos optimizar al máximo los hiperparámetros de este modelo base entrenando con la base congelada, y ya
posteriormente para el entrenamiento final probaremos un entrenamiento de dos etapas y otro con las últimas capas de la base descongeladas en todo momento
donde comrpobaremos que funciona mejor ajustando con unas pruebas el lr para la parte de la base descongelada.

En esta primera prueba se uso el weighted kl como perdida objetivo, por tanto en pruebas futuras los resultados probablemente parecerán peores.
"""

Hiperparámetros de los modelos ordenados del mejor al peor:
-----------------------------------------------------------------------------------------------
Resultado: 0.1445 | Duración: 0:17:29 | Parámetros: {'model_name': 'ConvNeXt_tiny', 'optimizer': 'Adam', 'learning rate': 0.001, 'size1': 1024, 'size2': 384}
Resultado: 0.1449 | Duración: 0:18:21 | Parámetros: {'model_name': 'ConvNeXt_tiny', 'optimizer': 'AdamW', 'learning rate': 0.001, 'size1': 896, 'size2': 320}
Resultado: 0.1462 | Duración: 0:15:54 | Parámetros: {'model_name': 'ConvNeXt_tiny', 'optimizer': 'Adam', 'learning rate': 0.001, 'size1': 1024, 'size2': 448}
Resultado: 0.1466 | Duración: 0:15:45 | Parámetros: {'model_name': 'ConvNeXt_tiny', 'optimizer': 'Adam', 'learning rate': 0.001, 'size1': 1024, 'size2': 320}
Resultado: 0.1469 | Duración: 0:23:42 | Parámetros: {'model_name': 'ConvNeXt_tiny', 'optimizer': 'Adam', 'learning rate': 0.001, 'size1': 1024, 'size2': 384}
Resultado: 0.1472 | Duración: 0:19:00 | Parámetros: {'

'"\nVemos claramente que la mejor base convolucional es ConvNeXt_tiny con los optimizadores Adam y AdamW entre los que podemos todavía hacer una comparativa más\nprofunda para elegir el mejor de los dos, el mejor learning rate con este modelo específico parece estar en torno al 0.001, en cuanto al tamaño de las capas\ntotalmente conectadas que tenemos añadirdas parece que los valores están en  torno al 1000 para la primera y el 300-400.\nPara el siguiente ajuste de hiperparámetros vamos a probar dentro de la familia ConvNeXt cual sería el mejor modelo y optimizaremos los hiperparámetros de\nestos para conseguir los mejores resultados, a priori parece que un modelo más potente de la misma familia debería funcionar mejor, pero es posible que sea\ndemasiado potente y se adapte demasiado a los ejemplos de entrenamiento generalizando mal posteriormente.\n\nEn esta primera prueba se uso el weighted kl como perdida objetivo, por tanto en pruebas futuras los resultados probablemente parecerán 

In [ ]:
""""!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!"""
### Vamos a hacer la segunda prueba de hiperparametros utilizando optuna ###
# Ahora que hemos visto que ConvNexT_tiny da buenos resultados, podemos centrarnos en ajustar sus hiperparámetros
from scripts_proporciones.models_training import complete_training
from scripts_proporciones.create_dataset import CustomImageDataset

# Elegimos la gpu si está disponible y si no la cpu
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Cargamos en data frames los conjuntos de datos ya divididos
train = pd.read_csv("./dataset_dividido/train.csv")
val = pd.read_csv("./dataset_dividido/val.csv")

# Creamos los datasets y dataloaders que vamos a usar para entrenar y validar
train_dataset = CustomImageDataset("./fotos_recortadas",train,True)
val_dataset = CustomImageDataset("./fotos_recortadas",val,True)

# Primeros valores de hiperparámetros a probar
def objective_optimize_CNN(trial):
    try:
        # Optimizador a utilizar
        optimizer = trial.suggest_categorical("optimizer", ["Adam","AdamW"])
        # Lr a utilizar (exploramos solo unos pocos posibles para agilizar el proceso de selección de base)
        lr = trial.suggest_float("learning rate", 1e-4, 1e-2, log=True)
        # Tamaños de las capas totalmente conectadas que añadimos al modelo de base (diferentes modelos rinden mejor con diferentes tamaños)
        size1 = trial.suggest_int("size1",768,1280,step=128)
        size2 = trial.suggest_int("size2",256,640,step=128)
        # Probamos tamaños de batch size
        batch_size = trial.suggest_categorical("batch size",[16,32,64,128])
        # Diferentes valores de dropout
        dropout = trial.suggest_categorical("dropout",[0.2,0.3,0.4,0.5])

        # Juntamos los hiperparámetros a optimizar"
        parameters = {"optimizer":optimizer,"learning rate":lr,"size1":size1,"size2":size2,"batch size":batch_size,"dropout":dropout}

        print("Empezando un nuevo intento con los siguientes parámetros:",parameters)
        # Esta función se llama al final de cada época y sirve para podar las ejecuciones menos prometedoras
        def pruning_callback(loss, epoch):
            trial.report(loss, epoch)
            if trial.should_prune():
                print("El trial no era prometedor y va a ser podado")
                raise optuna.exceptions.TrialPruned()

        train_dataloader = DataLoader(train_dataset,batch_size=batch_size,shuffle=True,pin_memory=True,num_workers=2)
        val_dataloader = DataLoader(val_dataset,batch_size=64,shuffle=False,pin_memory=True,num_workers=2)

        return complete_training("MRConvolutional","ConvNeXt_tiny",optimizer,train_dataloader,val_dataloader,lr1=lr,lr2=None,dropout=dropout,fine_tuning=False,
                                 size1=size1,size2=size2,patience1=10,patience2=None,max_epochs1=30,max_epochs2=None,label_smoothing=0.01,device=device,callback=pruning_callback)
    
    # Tratamiento de errores
    except optuna.exceptions.TrialPruned as e:
        raise
    except Exception as e:
        print("El trial falló, se evitarán esos hiperparámetros")
        print(e)
        raise  

# ------ Realizamos la búsqueda de hiperparámetros utilizando optuna y podando las combinaciones que no sean prometedoras
pruner = MedianPruner(n_startup_trials=15,n_warmup_steps=10)
study = optuna.create_study(
    direction="minimize",
    study_name="TFG_optimizar_CNN",
    storage="sqlite:///Hiperparametros_MR/optimizar_CNN.db",
    load_if_exists=True,
    pruner=pruner,
)
study.optimize(objective_optimize_CNN,n_trials=50)

In [21]:
# Importancia de hiperparámetros de proporciones
study = optuna.load_study(
    study_name="TFG_optimizar_CNN",
    storage="sqlite:///Hiperparametros_MR/optimizar_CNN.db"
)

# Evaluamos la importancia de cada hiperparámetro y la representamos
fig = vis.plot_param_importances(study)
fig.show()

In [ ]:
# Estudiamos los hiperparámetros con mejores resultados de la primera prueba y asi elegimos el mejor modelo base

# Mejores parámetros obtenidos en la primera prueba
study = optuna.load_study(
    study_name="TFG_optimizar_CNN",
    storage="sqlite:///Hiperparametros_MR/optimizar_CNN.db"
)
top_trials = sorted(
    [t for t in study.trials if t.state == TrialState.COMPLETE],
    key=lambda t: t.value
)[:20]
print("Hiperparámetros de los modelos ordenados del mejor al peor:")
print("-----------------------------------------------------------------------------------------------")
for i, trial in enumerate(top_trials, 1):
    print(f"Resultado: {trial.value:.4f}{"|":^3}Duración: {str(trial.duration).split(".")[0]}{"|":^3}Parámetros: {trial.params}")

"""
Estudiando los resultados y las importancias podemos sacar conclusiones sobre los hiperparámetros:
Primero destacar que el optimizador y los tamños de las capas totalmente conectadas no son especialmente importantes la variación de resultados,
nos quedaremos con 1152 para la primera capa y 384 para la segunda capa, ya que parecen funcionar bien, y para el optimizador nos quedamos con el
AdamW que es más moderno y suele funcionar mejor aunque en este caso no haya mucha diferencia respecto al Adam.
Por otro lado los hiperparámetros más importantes serían el batch size, el dropout y el learning rate. Empezando por dropout parece que 0.2 es el
valor que mejor funciona, podríamos probar incluso con 0.1 pero sería muy bajo y es interesante tener cierta cantidad de dropout para evitar
sobreajuste. El batch size que mejor funciona también es evidente en vista de los resultados siendo 64. El learning rate igual necesita algún ajuste
en la práctica cuando probemos a descongelar las últimas capas pero ya compararemos algunos valores discretos en concreto para ver si funcionan mejor,
pero en vista de los resultados parece que un valor de 2e-3 funciona bien aunque valores intermedios entre 1e-3 y 2e-3 también podrían darse.
"""

Hiperparámetros de los modelos ordenados del mejor al peor:
-----------------------------------------------------------------------------------------------
Resultado: 0.3623 | Duración: 0:24:23 | Parámetros: {'optimizer': 'Adam', 'learning rate': 0.001961819107194779, 'size1': 1152, 'size2': 384, 'batch size': 64, 'dropout': 0.2}
Resultado: 0.3627 | Duración: 0:23:58 | Parámetros: {'optimizer': 'AdamW', 'learning rate': 0.0018599934162353436, 'size1': 1152, 'size2': 384, 'batch size': 64, 'dropout': 0.2}
Resultado: 0.3637 | Duración: 0:24:23 | Parámetros: {'optimizer': 'AdamW', 'learning rate': 0.0018173908521553863, 'size1': 1152, 'size2': 512, 'batch size': 64, 'dropout': 0.2}
Resultado: 0.3647 | Duración: 0:24:36 | Parámetros: {'optimizer': 'AdamW', 'learning rate': 0.0016088886906419342, 'size1': 1152, 'size2': 512, 'batch size': 64, 'dropout': 0.2}
Resultado: 0.3668 | Duración: 0:24:35 | Parámetros: {'optimizer': 'Adam', 'learning rate': 0.0014755156760170058, 'size1': 1280, 'size

# Rendimiento en conjunto de test (Tarea de proporciones)

Vamos a crear dos baselines para comparar con los modelos, la primera predice siempre 0 para todas las clases y la segunda predice la proporción media de las clases para las muestras de train

In [3]:
from scripts_proporciones.create_dataset import CustomImageDataset

# Este primer modelo nos sirve para comprobar si nuestro propio modelo ignora ciertas clases por no afectar mucho a la pérdida total
class ZeroBaselineModel(nn.Module):
    def __init__(self):
        super().__init__()
    def forward(self,x,x_hist=None):
        # Usamos un vector de números muy pequeños para que al hacer exp(-1e9) el resultado se aproxime a 0
        return torch.full((x.shape[0],10),-9e8,device=x.device)
    @property
    def name(self):
        return "Zero Baseline"
    
# Este segundo modelo nos sirve para comprobar si nuestro propio modelo está aprendiendo características de la imagen o en cambio solo memoriza 
#  la distribución del conjunto de entrenamiento
class MeanBaselineModel(nn.Module):
    def __init__(self,labels_mean):
        super().__init__()
        # Vector con las medias de cada clase
        self.labels_mean = torch.log(labels_mean)
    def forward(self,x,x_hist=None):
        # Devolvemos el logaritmo de las medias puesto que después para mae se usa la exponencial para revertir el logaritmo
        return self.labels_mean.repeat(x.shape[0],1)
    @property
    def name(self):
        return "Mean Baseline"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# 1ª Baseline:
zeroBaseline = ZeroBaselineModel()

# 2ª Baseline:
# Dataset de entrenamiento utilizado    
train = pd.read_csv("./dataset_dividido/train.csv")
train_dataset = CustomImageDataset("./fotos_recortadas",train,True)
train_means = train_dataset.labels.mean(axis=0)


meanBaseline = MeanBaselineModel(torch.tensor(train_means,device=device))

In [4]:
# Cargamos en data frames los conjuntos de datos ya divididos
test = pd.read_csv("./dataset_dividido/test.csv")

# Creamos el datasets y dataloader que vamos a usar para comparar los modelos ya entrenados
test_dataset = CustomImageDataset("./fotos_recortadas",test,True,hist=True)
test_dataloader = DataLoader(test_dataset,batch_size=64,shuffle=False,pin_memory=True)

modelMRConvBase = MRConvolutionalModel("ConvNeXt_tiny",0.4,size1=640,size2=192).to(device)
modelMRHistogramConv = MRConvolutionalModel("ConvNeXt_tiny",0.4,size1=640,size2=192,use_histogram=True,num_bins=32).to(device)

# modelMRVisionT_Swin = MRVisionTransformer("Swin_V2_S",0.2,size1=512,size2=384).to(device)
# modelMRVisionT_Dino = MRVisionTransformer("DINOv2_ViT_B",0.2,size1=512,size2=384).to(device)

modelMRConvBase.load_state_dict(torch.load("Modelos/MODELO_CONVNEXT_TINY_SIN_SUAVIZAR_PESOS.pth",map_location=device))

modelMRHistogramConv.load_state_dict(torch.load("Modelos/MRConvolutional_Histogram_ConvNeXt_tiny_best_model.pth",map_location=device))
# modelMRVisionT_Swin.load_state_dict(torch.load("MRVisionT_Swin_V2_S_best_model.pth",map_location=device))
# modelMRVisionT_Dino.load_state_dict(torch.load("MRVisionT_DINOv2_ViT_B_best_model.pth",map_location=device))


models = [zeroBaseline,meanBaseline,modelMRConvBase,modelMRHistogramConv]#,modelMRVisionT_Swin,modelMRVisionT_Dino]

In [ ]:
import time
mae_loss_module = nn.L1Loss(reduction="sum")
kl_divergence_module = nn.KLDivLoss(reduction="sum")
target_cols = ['n. noltei','z. marina','g. vermiculophylla','sedimento','arena','roca','algas verdes','algas pardas','algas rojas','microfitobentos']

with torch.no_grad():
    for model in models:
        val_mae_loss = 0
        val_kl_divergence = 0
        total_samples = 0

        class_mae_loss = 0
        present_class_mae_loss = torch.zeros(10,device=device)
        present_class_count = torch.zeros(10,device=device)

        confusion_matrix = torch.zeros((10,10),device=device)

        model.eval()
        start=time.time()

        for data_inputs,data_labels,data_hist in test_dataloader:
            data_inputs = data_inputs.to(device)
            data_labels = data_labels.to(device)
            if data_hist is not None:
                data_hist = data_hist.to(device)
            mask = data_labels>0

            total_samples += data_inputs.size(0)

            # Vamos a evitar que las etiquetas tengan el valor exacto 0 para evitar que la divergencia kl colapse
            smooth_labels = data_labels*(1-0.01)+0.01/10
            log_preds = model(data_inputs,data_hist)
            # Calcular el valor de la función de pérdida mae y la divergencia kl
            y_pred = torch.exp(log_preds)

            # Calculamos las pérdidas
            val_mae_loss += mae_loss_module(y_pred,data_labels).item()
            val_kl_divergence += kl_divergence_module(log_preds,smooth_labels).item()
            # Calculamos el MAE por cada clase individual
            class_mae_loss += torch.sum(torch.abs(y_pred-data_labels),dim=0)

            # Vamos a comprobar el MAE por clase individual solo considerando las fotos en las que aparezca dicha clase
            for i in range(len(present_class_mae_loss)):
                mask_i = mask[:,i]
                if mask_i.any():
                    present_class_mae_loss[i] += torch.sum(torch.abs(y_pred[mask_i,i]-data_labels[mask_i,i]))
                    present_class_count[i] += mask_i.sum()

            # Calculamos las predicciones que se quedaron cortas de proporción y las que se pasaron de proporción
            # Usamos clamp para evitar posibles valores negativos
            deficit = torch.clamp(data_labels - y_pred,min=0)
            exceso  = torch.clamp(y_pred - data_labels,min=0)
            # Se hace el producto exterior por muestra para obtener la matriz de confusión
            # deficit[:,i]*exceso[:,j] = (batch,i,j)
            confusion_matrix += torch.einsum('bi,bj->ij',deficit,exceso)

        # Normalizamos la matriz de confusión fila por fila
        row_sums = confusion_matrix.sum(dim=1,keepdim=True)
        confusion_norm = torch.round(confusion_matrix/row_sums,decimals=2)
        confusion_m = pd.DataFrame(confusion_norm.cpu().numpy(),index=target_cols, columns=target_cols)

        # Construimos una estructura del MAE por clase para imprimirlo
        class_mae = " | ".join([str(round(i,4)) for i in (class_mae_loss/total_samples).tolist()])
        present_class_mae = " | ".join([str(round(i,4)) for i in (present_class_mae_loss/present_class_count).tolist()])

        # Imprimimos resultados para comparar
        print("Modelo de red multiregresión %s.\n"
             "Divergencia KL: %.4f , MAE Loss: %.4f\n"
             "MAE por clases: %s\n"
             "MAE por clases: %s - solo en imágenes en las que aparecen\n"
             "Tiempo de ejecución: %.2fs"%
             (model.name,val_kl_divergence/total_samples,val_mae_loss/(total_samples*10),class_mae,present_class_mae,time.time()-start))
        

        # Imprimir top confusiones
        display(confusion_m)
        print("")

Modelo de red multiregresión Zero Baseline.
Divergencia KL: 899999994.0858 , MAE Loss: 0.1000
MAE por clases: 0.3936 | 0.0145 | 0.015 | 0.2894 | 0.121 | 0.0049 | 0.1185 | 0.0393 | 0.0004 | 0.0035
MAE por clases: 0.6343 | 0.6286 | 0.228 | 0.5127 | 0.6664 | 0.1838 | 0.374 | 0.5174 | 0.0367 | 0.214 - solo en imágenes en las que aparecen
Tiempo de ejecución: 2.69s


,n. noltei,z. marina,g. vermiculophylla,sedimento,arena,roca,algas verdes,algas pardas,algas rojas,microfitobentos
n. noltei,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
z. marina,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
g. vermiculophylla,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
sedimento,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
arena,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
roca,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
algas verdes,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
algas pardas,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
algas rojas,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
microfitobentos,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



tensor([[0.3822, 0.0127, 0.0206, 0.2826, 0.1282, 0.0110, 0.1164, 0.0400, 0.0039,
         0.0023],
        [0.3822, 0.0127, 0.0206, 0.2826, 0.1282, 0.0110, 0.1164, 0.0400, 0.0039,
         0.0023],
        [0.3822, 0.0127, 0.0206, 0.2826, 0.1282, 0.0110, 0.1164, 0.0400, 0.0039,
         0.0023],
        [0.3822, 0.0127, 0.0206, 0.2826, 0.1282, 0.0110, 0.1164, 0.0400, 0.0039,
         0.0023],
        [0.3822, 0.0127, 0.0206, 0.2826, 0.1282, 0.0110, 0.1164, 0.0400, 0.0039,
         0.0023],
        [0.3822, 0.0127, 0.0206, 0.2826, 0.1282, 0.0110, 0.1164, 0.0400, 0.0039,
         0.0023],
        [0.3822, 0.0127, 0.0206, 0.2826, 0.1282, 0.0110, 0.1164, 0.0400, 0.0039,
         0.0023],
        [0.3822, 0.0127, 0.0206, 0.2826, 0.1282, 0.0110, 0.1164, 0.0400, 0.0039,
         0.0023],
        [0.3822, 0.0127, 0.0206, 0.2826, 0.1282, 0.0110, 0.1164, 0.0400, 0.0039,
         0.0023],
        [0.3822, 0.0127, 0.0206, 0.2826, 0.1282, 0.0110, 0.1164, 0.0400, 0.0039,
         0.0023],
        [

,n. noltei,z. marina,g. vermiculophylla,sedimento,arena,roca,algas verdes,algas pardas,algas rojas,microfitobentos
n. noltei,0.00,0.02,0.04,0.41,0.23,0.02,0.19,0.07,0.01,0.0
z. marina,0.37,0.00,0.02,0.26,0.16,0.01,0.12,0.05,0.00,0.0
g. vermiculophylla,0.42,0.02,0.00,0.20,0.18,0.02,0.10,0.06,0.01,0.0
sedimento,0.49,0.02,0.03,0.00,0.21,0.02,0.16,0.06,0.01,0.0
arena,0.44,0.02,0.03,0.33,0.00,0.01,0.12,0.05,0.00,0.0
roca,0.40,0.01,0.02,0.27,0.14,0.00,0.11,0.04,0.00,0.0
algas verdes,0.39,0.02,0.03,0.31,0.17,0.02,0.00,0.05,0.01,0.0
algas pardas,0.44,0.02,0.02,0.23,0.15,0.01,0.11,0.00,0.00,0.0
algas rojas,0.31,0.02,0.03,0.22,0.20,0.02,0.13,0.06,0.00,0.0
microfitobentos,0.53,0.02,0.03,0.02,0.17,0.02,0.16,0.06,0.01,0.0



Modelo de red multiregresión MRConvolutional_ConvNeXt_tiny.
Divergencia KL: 0.3028 , MAE Loss: 0.0456
MAE por clases: 0.0903 | 0.0136 | 0.0317 | 0.1369 | 0.0599 | 0.0136 | 0.0598 | 0.0181 | 0.0166 | 0.0155
MAE por clases: 0.1358 | 0.1142 | 0.1624 | 0.1895 | 0.1498 | 0.0564 | 0.1459 | 0.1271 | 0.0061 | 0.1071 - solo en imágenes en las que aparecen
Tiempo de ejecución: 4.89s


,n. noltei,z. marina,g. vermiculophylla,sedimento,arena,roca,algas verdes,algas pardas,algas rojas,microfitobentos
n. noltei,0.00,0.05,0.19,0.26,0.12,0.03,0.22,0.04,0.06,0.04
z. marina,0.00,0.00,0.14,0.24,0.12,0.03,0.32,0.03,0.06,0.06
g. vermiculophylla,0.18,0.07,0.00,0.15,0.05,0.03,0.21,0.04,0.23,0.03
sedimento,0.12,0.05,0.10,0.00,0.38,0.07,0.09,0.09,0.06,0.05
arena,0.09,0.02,0.03,0.61,0.00,0.05,0.11,0.01,0.03,0.06
roca,0.03,0.05,0.05,0.23,0.10,0.00,0.19,0.14,0.12,0.10
algas verdes,0.16,0.04,0.09,0.34,0.09,0.09,0.00,0.09,0.05,0.06
algas pardas,0.02,0.11,0.30,0.10,0.06,0.06,0.12,0.00,0.17,0.06
algas rojas,0.02,0.07,0.06,0.47,0.20,0.06,0.03,0.03,0.00,0.05
microfitobentos,0.01,0.04,0.01,0.57,0.13,0.06,0.08,0.06,0.03,0.00



Modelo de red multiregresión MRConvolutional_Histogram_ConvNeXt_tiny.
Divergencia KL: 0.2245 , MAE Loss: 0.0368
MAE por clases: 0.0772 | 0.0063 | 0.0206 | 0.1248 | 0.0494 | 0.0069 | 0.054 | 0.0156 | 0.0052 | 0.0083
MAE por clases: 0.1143 | 0.0736 | 0.1423 | 0.1565 | 0.1636 | 0.0536 | 0.1387 | 0.1288 | 0.0295 | 0.1713 - solo en imágenes en las que aparecen
Tiempo de ejecución: 5.05s


,n. noltei,z. marina,g. vermiculophylla,sedimento,arena,roca,algas verdes,algas pardas,algas rojas,microfitobentos
n. noltei,0.00,0.03,0.12,0.45,0.07,0.01,0.25,0.04,0.02,0.01
z. marina,0.06,0.00,0.06,0.30,0.09,0.01,0.37,0.04,0.02,0.03
g. vermiculophylla,0.32,0.03,0.00,0.31,0.04,0.01,0.18,0.03,0.08,0.02
sedimento,0.20,0.02,0.08,0.00,0.39,0.05,0.13,0.08,0.02,0.03
arena,0.06,0.01,0.01,0.78,0.00,0.03,0.07,0.01,0.01,0.02
roca,0.10,0.03,0.02,0.38,0.06,0.00,0.18,0.16,0.04,0.03
algas verdes,0.21,0.01,0.05,0.45,0.07,0.05,0.00,0.10,0.01,0.03
algas pardas,0.08,0.07,0.19,0.45,0.03,0.02,0.11,0.00,0.04,0.02
algas rojas,0.03,0.05,0.03,0.72,0.07,0.01,0.03,0.03,0.00,0.03
microfitobentos,0.01,0.01,0.01,0.74,0.05,0.07,0.06,0.04,0.01,0.00


# Ejemplos individuales

In [ ]:
target_cols = ['n. noltei','z. marina','g. vermiculophylla','sedimento','arena','roca','algas verdes','algas pardas','algas rojas','microfitobentos']

model = modelMRConvBase
test = pd.read_csv("./dataset_dividido/test.csv")
# Creamos el datasets y dataloader que vamos a usar para comparar los modelos ya entrenados
test_dataset = CustomImageDataset("./fotos_recortadas",test,True,hist=True)
test_dataloader = DataLoader(test_dataset,batch_size=16,shuffle=False,pin_memory=True)

fotos = test["foto"][:16]

with torch.no_grad():
    data_inputs,data_labels,_ = next(iter(test_dataloader))
    data_inputs = data_inputs.to(device)
    data_labels = data_labels.to(device)

    model.eval()

    log_preds = model(data_inputs)
    y_pred = torch.exp(log_preds)

    for i in range(len(data_labels)):
        img = cv2.cvtColor(cv2.imread(f"./fotos_recortadas/{fotos[i]}"),cv2.COLOR_BGR2RGB)

        plt.figure(figsize=(6,6))
        plt.imshow(img)
        plt.title(fotos[i])
        print(fotos[i])
        plt.axis('off')
        plt.show()

        etiqueta = data_labels.cpu().numpy()[i]
        predicción = np.round(y_pred.cpu().numpy()[i],2)
        logaritmo = np.round(log_preds.cpu().numpy()[i],2)

        df = pd.DataFrame([etiqueta,logaritmo,predicción], index=["Etiqueta","Logaritmo","Predicción"], columns=target_cols)
        display(df)

## Comprobamos el rendimiento con los nuevos ejemplos

### Preprocesamos el nuevo dataset

In [4]:
# Utilizamos el nuevo conjunto de datos para comprobar como funcionan los modelos con datos nunca vistos durante el entrenamiento

# Cargamos el excel usando pandas y nos quedamos solo con las columnas de las etiquetas y la del nombre de la foto
# Además vamos a cambiar el nombre de las columnas con el mismo que se usó en el anterior dataset
target_cols = ['n. noltei','z. marina','g. vermiculophylla','sedimento','arena','roca','algas verdes','algas pardas','algas rojas','microfitobentos']

cols = [0]+list(range(18,28))
labels_route = "4_new_dataset/1_SEATRUTH_table.xlsx"
df = pd.read_excel(labels_route,usecols=cols)
# Cambiamos el nombre de las columnas para que coincida con el nombre que usamos en el anterior dataset
df.columns =  ["foto"]+target_cols
# Pasamos el campo foto al formato de las fotografías con su extensión
df["foto"]= df["foto"]+".jpg"

# Nos quedamos solo con los registros que tengan alguna de las fotos recortadas en su campo "foto"
# Esto nos ahorra además tener que eliminar los registros que tengan un valor vacío en la columna correspondiente
cropped_photos = [n for n in os.listdir("nuevo_dataset_recortado") if n.endswith((".jpg",".jpeg",".png"))]
df = df[df["foto"].isin(cropped_photos)]

# Vamos a pasar las columnas a float para evitar futuros errores
df[target_cols] = df[target_cols].astype(float)
df

,foto,n. noltei,z. marina,g. vermiculophylla,sedimento,arena,roca,algas verdes,algas pardas,algas rojas,microfitobentos
5855,20250729_123543_PONTEVEDRA.jpg,100.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5856,20250729_124154_PONTEVEDRA.jpg,65.0,0.0,0.0,20.0,0.0,0.0,15.0,0.0,0.0,0.0
5857,20250729_124306_PONTEVEDRA.jpg,65.0,0.0,0.0,20.0,0.0,0.0,15.0,0.0,0.0,0.0
5858,20250729_124358_PONTEVEDRA.jpg,100.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5859,20250729_124529_PONTEVEDRA.jpg,20.0,0.0,0.0,75.0,0.0,0.0,5.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...
6170,20260220_121832_PONTEVEDRA.jpg,0.0,25.0,0.0,60.0,10.0,0.0,0.0,0.0,0.0,5.0
6171,20260220_122955_PONTEVEDRA.jpg,0.0,30.0,0.0,60.0,5.0,0.0,0.0,0.0,0.0,5.0
6172,20260303_102628_PONTEVEDRA.jpg,10.0,0.0,0.0,50.0,40.0,0.0,0.0,0.0,0.0,0.0
6175,20260303_104703_PONTEVEDRA.jpg,0.0,20.0,0.0,20.0,60.0,0.0,0.0,0.0,0.0,0.0


In [25]:
# Hacemos una comprobación rápida para ver que no hay duplicados
dups=df[df.duplicated(subset=["foto"],keep=False)]
dups


,foto,n. noltei,z. marina,g. vermiculophylla,sedimento,arena,roca,algas verdes,algas pardas,algas rojas,microfitobentos


In [ ]:
# Primero hacemos una comprobación rápida para ver si hay valores faltantes en nuestras columnas objetivo
print(df[df[target_cols].isna().any(axis=1)])
# Nos devuelve un empty data frame porque no hay valores faltantes

# Sabemos que nuestras columnas objetivo incluyen el porcentaje de cada clase que hay presente en la imagen
# Por tanto la suma de las columnas para un registro siempre debería sumar 100, vamos a comprobar si hay 
#  registros en los que la suma de o más o menos de 100 y estudiaremos el posible por qué y buscaremos una solución
wrong_values =df[df.loc[:,target_cols].sum(axis=1)!=100]
wrong_values

Empty DataFrame
Columns: [foto, n. noltei, z. marina, g. vermiculophylla, sedimento, arena, roca, algas verdes, algas pardas, algas rojas, microfitobentos]
Index: []


,foto,n. noltei,z. marina,g. vermiculophylla,sedimento,arena,roca,algas verdes,algas pardas,algas rojas,microfitobentos
5870,20250729_133810_PONTEVEDRA.jpg,25.0,25.0,0.0,25.0,20.0,0.0,0.0,1111.0,0.0,5.0
5875,20250729_135535_PONTEVEDRA.jpg,0.0,25.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5888,20250812_122701_AROUSA.jpg,0.0,50.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5889,20250812_122857_AROUSA.jpg,0.0,50.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5890,20250812_123048_AROUSA.jpg,25.0,25.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5891,20250812_123215_AROUSA.jpg,75.0,10.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5892,20250812_123737_AROUSA.jpg,95.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5893,20250812_123948_AROUSA.jpg,0.0,80.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5894,20250812_124335_AROUSA.jpg,95.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5895,20250812_133045_AROUSA.jpg,80.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [27]:
# Está claro que el primer registro tiene un valor incorrecto en la columna de algas pardas, por lo que lo ponemos a 0
df.loc[5870,"algas pardas"] = 0
wrong_values =df[df.loc[:,target_cols].sum(axis=1)!=100]
wrong_values

,foto,n. noltei,z. marina,g. vermiculophylla,sedimento,arena,roca,algas verdes,algas pardas,algas rojas,microfitobentos
5875,20250729_135535_PONTEVEDRA.jpg,0.0,25.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5888,20250812_122701_AROUSA.jpg,0.0,50.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5889,20250812_122857_AROUSA.jpg,0.0,50.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5890,20250812_123048_AROUSA.jpg,25.0,25.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5891,20250812_123215_AROUSA.jpg,75.0,10.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5892,20250812_123737_AROUSA.jpg,95.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5893,20250812_123948_AROUSA.jpg,0.0,80.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5894,20250812_124335_AROUSA.jpg,95.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5895,20250812_133045_AROUSA.jpg,80.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5950,20250923_111631_PONTEVEDRA.jpg,0.0,0.0,0.0,50.0,0.0,0.0,30.0,10.0,0.0,0.0


In [28]:
"""
Lo que haremos ahora es comprobar para los demás ejemplos mal etiquetados, comprobar si al menos los elementos que las etiquetas
están indicando presentes en las fotos realmente lo están y veremos si la proporción que proponen tiene sentido, es decir,
si pone que hay 80 arena y 10 alga verde, pero después la foto está llena de algas verdes y casi no hay arena pués no tendrá
sentido, por el contrario si pone 40 arena y 40 alga verde, y después en la imagen hay mitad arena mitad alga verde, es posible
que los valores reales sean 50|50 y la proporción relativa sea correcta entre ellos, otro caso que podría darse es que ponga
40 arena y 40 alga verde, pero luego en la imagen haya algo de alga roja que faltaba por identificarse.
Si lo que se da es el primer o tercer caso, lo más probable es que esos registros sean inservibles y tengamos que eliminarlos.
"""
"""
Después de una primera inspección por encima podemos sacar una conclusion, que en los casos donde solo hay un elemento visible
según la etiqueta parece que faltan otros elementos por mencionar, en la imagen se ven claramente varias cosas diferentes pero en la etiqueta
solo se recoge una de ellas con su proporción, es decir que las etiquetas están incompletas puesto que les faltan elementos que no consideran.
Por tanto, vamos a empezar por el caso de que solo se indique un elemento en la imagen.
"""
one_target=wrong_values[((wrong_values.loc[:,target_cols]>0).sum(axis=1))==1]
one_target

,foto,n. noltei,z. marina,g. vermiculophylla,sedimento,arena,roca,algas verdes,algas pardas,algas rojas,microfitobentos
5875,20250729_135535_PONTEVEDRA.jpg,0.0,25.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5888,20250812_122701_AROUSA.jpg,0.0,50.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5889,20250812_122857_AROUSA.jpg,0.0,50.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5892,20250812_123737_AROUSA.jpg,95.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5893,20250812_123948_AROUSA.jpg,0.0,80.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5894,20250812_124335_AROUSA.jpg,95.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5895,20250812_133045_AROUSA.jpg,80.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5999,20250923_122226_PONTEVEDRA.jpg,95.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [29]:
"""
La revisión de estos ejemplos es muy rápida porque tendrían que ser fotos uniformes con un único elemento presente, sin embargo,
al revisar las fotografías podemos confirmar la hipótesis inicial de que las etiquetas están incompletas puesto que a simple
vista somos capaces de identificar la presencia de múltiples otros elementos que no indican.
De esta manera vamos a descartar todos los registros en los que se dé esta situación de nuestro conjunto de datos y después
revisaremos los ejemplos con porcentajes incorrectos pero varios elementos recogidos.
"""
df=df.drop(index=one_target.index)

In [30]:
wrong_values = df[df.loc[:,target_cols].sum(axis=1)!=100]
wrong_values

,foto,n. noltei,z. marina,g. vermiculophylla,sedimento,arena,roca,algas verdes,algas pardas,algas rojas,microfitobentos
5890,20250812_123048_AROUSA.jpg,25.0,25.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5891,20250812_123215_AROUSA.jpg,75.0,10.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5950,20250923_111631_PONTEVEDRA.jpg,0.0,0.0,0.0,50.0,0.0,0.0,30.0,10.0,0.0,0.0
5982,20250923_120014_PONTEVEDRA.jpg,80.0,0.0,0.0,30.0,0.0,0.0,0.0,0.0,0.0,0.0
6010,20250923_133134_PONTEVEDRA.jpg,15.0,0.0,0.0,60.0,0.0,0.0,15.0,0.0,0.0,0.0
6011,20250923_133253_PONTEVEDRA.jpg,0.0,0.0,0.0,25.0,25.0,0.0,25.0,15.0,0.0,15.0
6039,20251009_142627_PONTEVEDRA.jpg,80.0,0.0,0.0,15.0,0.0,0.0,0.0,0.0,0.0,0.0


In [31]:
"""
Como son pocas imágenes podemos revisarlas indicidualmente y modificarlas o eliminarlas una por una.
Haciendo una revisión uno a uno de los registros con porcentajes incorrectos se puede ver lo siguiente:
    - La primera foto parece tener más elementos y es dificil aproximar la estimación
    - En la segunda foto parece faltar un 15 de arena
    - En la tercera foto es dificil estimar las proporciones faltantes
    - En la cuarta foto parece sobrar 10 de n.noltei
    - En la quinta foto parece faltar 10 de sedimento
    - En la sexta foto es dificil estimar las proporciones faltantes
    - En la septima foto parece faltar 5 de n.noltei
En base a estas observaciones podemos eliminar las filas problemáticas y modificar los valores de las otras fila para que coincidan con el contenido de las fotos
"""
df=df.drop(index=[5890,5950,6011])
df.loc[5891,"arena"] = 15
df.loc[5982,"n. noltei"] = 70
df.loc[6010,"sedimento"] = 70
df.loc[6039,"n. noltei"] = 85

# Comprobamos que todos los valores incorrectos estén arreglados
wrong_values = df[df.loc[:,target_cols].sum(axis=1)!=100]
wrong_values

,foto,n. noltei,z. marina,g. vermiculophylla,sedimento,arena,roca,algas verdes,algas pardas,algas rojas,microfitobentos


In [ ]:
# Como podemos ver no queda ningún registro que tenga incoherencias en las sumas y con esto concluimos el preprocesado.
# Ahora lo que hacemos es guardar el data frame limpio con las columnas que nos interesan
# df[target_cols+["foto"]].to_csv("etiquetas_nuevas_fotos.csv",index=False)

In [32]:
"""
Vamos a estudiar la presencia de cada clase en las fotografías para comprobar si las clases están desbalanceadas y nos puede convenir darles un tratamiento diferente.
Como podemos ver que hay varias clases que aparecen en una minoría de imagenes nos interesa tener cuidado con esos elementos puesto que nuestro modelo puede aprender
a predecir siempre cero para ellos y en la mayoría de imágenes sería correcto y por tanto no sería útil a la hora de predecir dichas clases.
Podría ser buena idea aplicar data augmentation a las imágenes pero al tener tan pocas fotos con presencia de estas clases comparado con las imágenes totales al final
o la variedad no será muy buena y tendremos muchas fotos "repetidas" con pequeños cambios, o si usamos menos fotos originales estamos desaprovechando etiquetas que
tenemos disponibles
"""
# Comprobamos el número de fotos en las que aparece cada clase
print("Presencia en fotos")
print((df[target_cols]>0).sum())

# Comprobamos la proporción de fotos en las que aparece cada imagen
print("\nPorcentaje de presencia en fotos")
print(round((df[target_cols]>0).sum()/len(df)*100,2))

# Comprobar que peso equipararía las clases para el entrenamiento

Presencia en fotos
n. noltei             97
z. marina             20
g. vermiculophylla     1
sedimento             90
arena                 24
roca                   3
algas verdes          48
algas pardas          14
algas rojas            1
microfitobentos       18
dtype: int64

Porcentaje de presencia en fotos
n. noltei             75.19
z. marina             15.50
g. vermiculophylla     0.78
sedimento             69.77
arena                 18.60
roca                   2.33
algas verdes          37.21
algas pardas          10.85
algas rojas            0.78
microfitobentos       13.95
dtype: float64


In [ ]:
# Vamos a estudiar las porporciones generales en las imágenes para las diferentes clases
fig = go.Figure()

# Agrupamos las clases en una misma columna
df_long = df[target_cols].melt(var_name="clase",value_name="proporcion")

fig = px.box(
    df_long,
    x="clase",
    y="proporcion",
    color="clase",
    points="outliers",
    title="Distribución de proporciones por clase",
)

fig.update_layout(
    yaxis=dict(title="Proporción por foot",range=[0, 100]),
    xaxis=dict(title="Clases"),
    height=550,
    showlegend=False,
)

# Visualizamos el diagrama de cajas general
fig.show()

In [36]:
# Ahora vamos a explorar las proporciones de las clases solo para las imágenes en las que estan presentes
fig = go.Figure()

# Agrupamos las clases en una misma columna y nos quedamos con los casos donde su proporción sea mayor que 0
df_long = df[target_cols].melt(var_name="clase",value_name="proporcion")
df_long = df_long[df_long["proporcion"]>0]

fig = px.box(
    df_long,
    x="clase",
    y="proporcion",
    color="clase",
    points="outliers",
    title="Distribución de proporciones por clase",
)

fig.update_layout(
    yaxis=dict(title="Proporción por foto",range=[0, 100]),
    xaxis=dict(title="Clases"),
    height=550,
    showlegend=False,
    font=dict(size=20)
)

# Dibujamos el diagrama de cajas considerando para cada clase las imágenes en las que aparecen
fig.show()

### Pasamos el dataset al modelo

In [7]:
from scripts_proporciones.create_dataset import CustomImageDataset

# Este primer modelo nos sirve para comprobar si nuestro propio modelo ignora ciertas clases por no afectar mucho a la pérdida total
class ZeroBaselineModel(nn.Module):
    def __init__(self):
        super().__init__()
    def forward(self,x,x_hist=None):
        # Usamos un vector de números muy pequeños para que al hacer exp(-1e9) el resultado se aproxime a 0
        return torch.full((x.shape[0],10),-9e8,device=x.device)
    @property
    def name(self):
        return "Zero Baseline"
    
# Este segundo modelo nos sirve para comprobar si nuestro propio modelo está aprendiendo características de la imagen o en cambio solo memoriza 
#  la distribución del conjunto de entrenamiento
class MeanBaselineModel(nn.Module):
    def __init__(self,labels_mean):
        super().__init__()
        # Vector con las medias de cada clase
        self.labels_mean = torch.log(labels_mean)
    def forward(self,x,x_hist=None):
        # Devolvemos el logaritmo de las medias puesto que después para mae se usa la exponencial para revertir el logaritmo
        return self.labels_mean.repeat(x.shape[0],1)
    @property
    def name(self):
        return "Mean Baseline"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# 1ª Baseline:
zeroBaseline = ZeroBaselineModel()

# 2ª Baseline:
# Dataset de entrenamiento utilizado    
train = pd.read_csv("./dataset_dividido/train.csv")
train_dataset = CustomImageDataset("./fotos_recortadas",train,True)
train_means = train_dataset.labels.mean(axis=0)

meanBaseline = MeanBaselineModel(torch.tensor(train_means,device=device))

In [8]:
# Cargamos el nuevo dataset para las pruebas
test = pd.read_csv("./etiquetas_nuevas_fotos.csv")

# Creamos el datasets y dataloader que vamos a usar para comparar los modelos ya entrenados
test_dataset = CustomImageDataset("./nuevo_dataset_recortado",test,True,hist=True)
test_dataloader = DataLoader(test_dataset,batch_size=64,shuffle=False,pin_memory=True)

modelMRConvBase = MRConvolutionalModel("ConvNeXt_tiny",0.4,size1=640,size2=192).to(device)
modelMRHistogramConv = MRConvolutionalModel("ConvNeXt_tiny",0.4,size1=640,size2=192,use_histogram=True,num_bins=32).to(device)

modelMRConvBase.load_state_dict(torch.load("Modelos/MRConvolutional_ConvNeXt_tiny_best_model.pth",map_location=device))
modelMRHistogramConv.load_state_dict(torch.load("Modelos/MRConvolutional_Histogram_ConvNeXt_tiny_best_model.pth",map_location=device))

models = [zeroBaseline,meanBaseline,modelMRConvBase,modelMRHistogramConv]

In [9]:
import time
mae_loss_module = nn.L1Loss(reduction="sum")
kl_divergence_module = nn.KLDivLoss(reduction="sum")
target_cols = ['n. noltei','z. marina','g. vermiculophylla','sedimento','arena','roca','algas verdes','algas pardas','algas rojas','microfitobentos']

with torch.no_grad():
    for model in models:
        val_mae_loss = 0
        val_kl_divergence = 0
        total_samples = 0

        class_mae_loss = 0
        present_class_mae_loss = torch.zeros(10,device=device)
        present_class_count = torch.zeros(10,device=device)

        confusion_matrix = torch.zeros((10,10),device=device)

        model.eval()
        start=time.time()

        for data_inputs,data_labels,data_hist in test_dataloader:
            data_inputs = data_inputs.to(device)
            data_labels = data_labels.to(device)
            if data_hist is not None:
                data_hist = data_hist.to(device)
            mask = data_labels>0

            total_samples += data_inputs.size(0)

            # Vamos a evitar que las etiquetas tengan el valor exacto 0 para evitar que la divergencia kl colapse
            smooth_labels = data_labels*(1-0.01)+0.01/10
            log_preds = model(data_inputs,data_hist)
            # Calcular el valor de la función de pérdida mae y la divergencia kl
            y_pred = torch.exp(log_preds)

            # Calculamos las pérdidas
            val_mae_loss += mae_loss_module(y_pred,data_labels).item()
            val_kl_divergence += kl_divergence_module(log_preds,smooth_labels).item()
            # Calculamos el MAE por cada clase individual
            class_mae_loss += torch.sum(torch.abs(y_pred-data_labels),dim=0)

            # Vamos a comprobar el MAE por clase individual solo considerando las fotos en las que aparezca dicha clase
            for i in range(len(present_class_mae_loss)):
                mask_i = mask[:,i]
                if mask_i.any():
                    present_class_mae_loss[i] += torch.sum(torch.abs(y_pred[mask_i,i]-data_labels[mask_i,i]))
                    present_class_count[i] += mask_i.sum()

            # Calculamos las predicciones que se quedaron cortas de proporción y las que se pasaron de proporción
            # Usamos clamp para evitar posibles valores negativos
            deficit = torch.clamp(data_labels - y_pred,min=0)
            exceso  = torch.clamp(y_pred - data_labels,min=0)
            # Producto exterior por muestra: deficit[:,i] * exceso[:,j] → (batch, i, j)
            # Suma sobre el batch para acumular en la matriz
            confusion_matrix += torch.einsum('bi,bj->ij', deficit, exceso)

        row_sums = confusion_matrix.sum(dim=1, keepdim=True)
        confusion_norm = torch.round(confusion_matrix / row_sums, decimals=2)
        confusion_m = pd.DataFrame(confusion_norm.cpu().numpy(), index=target_cols, columns=target_cols)

        # Construimos una estructura del MAE por clase para imprimirlo
        class_mae = " | ".join([str(round(i,4)) for i in (class_mae_loss/total_samples).tolist()])
        present_class_mae = " | ".join([str(round(i,4)) for i in (present_class_mae_loss/present_class_count).tolist()])

        # Imprimimos resultados para comparar
        print("Modelo de red multiregresión %s.\n"
             "Divergencia KL: %.4f , MAE Loss: %.4f\n"
             "MAE por clases: %s\n"
             "MAE por clases: %s - solo en imágenes en las que aparecen\n"
             "Tiempo de ejecución: %.2fs"%
             (model.name,val_kl_divergence/total_samples,val_mae_loss/(total_samples*10),class_mae,present_class_mae,time.time()-start))
        

        # Imprimir top confusiones
        display(confusion_m)
        print("")

Modelo de red multiregresión Zero Baseline.
Divergencia KL: 900000032.7442 , MAE Loss: 0.1000
MAE por clases: 0.524 | 0.0391 | 0.0008 | 0.2389 | 0.0756 | 0.0085 | 0.0773 | 0.0283 | 0.0008 | 0.0067
MAE por clases: 0.6968 | 0.2525 | 0.1 | 0.3424 | 0.4062 | 0.3667 | 0.2077 | 0.2607 | 0.1 | 0.0483 - solo en imágenes en las que aparecen
Tiempo de ejecución: 1.44s


,n. noltei,z. marina,g. vermiculophylla,sedimento,arena,roca,algas verdes,algas pardas,algas rojas,microfitobentos
n. noltei,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
z. marina,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
g. vermiculophylla,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
sedimento,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
arena,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
roca,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
algas verdes,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
algas pardas,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
algas rojas,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
microfitobentos,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Modelo de red multiregresión Mean Baseline.
Divergencia KL: 0.8903 , MAE Loss: 0.1080
MAE por clases: 0.3959 | 0.0479 | 0.0211 | 0.2385 | 0.1591 | 0.019 | 0.1255 | 0.0596 | 0.0046 | 0.0084
MAE por clases: 0.4004 | 0.2398 | 0.0794 | 0.2193 | 0.2939 | 0.3557 | 0.1409 | 0.2207 | 0.0961 | 0.046 - solo en imágenes en las que aparecen
Tiempo de ejecución: 1.34s


,n. noltei,z. marina,g. vermiculophylla,sedimento,arena,roca,algas verdes,algas pardas,algas rojas,microfitobentos
n. noltei,0.00,0.02,0.04,0.42,0.24,0.02,0.18,0.07,0.01,0.0
z. marina,0.52,0.00,0.03,0.14,0.11,0.02,0.12,0.06,0.01,0.0
g. vermiculophylla,0.54,0.00,0.00,0.26,0.18,0.02,0.00,0.00,0.01,0.0
sedimento,0.48,0.02,0.04,0.00,0.19,0.02,0.17,0.07,0.01,0.0
arena,0.47,0.01,0.03,0.26,0.00,0.02,0.16,0.05,0.00,0.0
roca,0.48,0.02,0.03,0.30,0.16,0.00,0.02,0.00,0.00,0.0
algas verdes,0.48,0.02,0.03,0.19,0.21,0.02,0.00,0.04,0.01,0.0
algas pardas,0.52,0.02,0.03,0.22,0.18,0.01,0.02,0.00,0.01,0.0
algas rojas,0.37,0.00,0.03,0.37,0.00,0.01,0.15,0.05,0.00,0.0
microfitobentos,0.53,0.01,0.03,0.09,0.09,0.02,0.16,0.06,0.01,0.0



Modelo de red multiregresión MRConvolutional_ConvNeXt_tiny.
Divergencia KL: 0.3348 , MAE Loss: 0.0374
MAE por clases: 0.0946 | 0.0243 | 0.0059 | 0.1023 | 0.0643 | 0.0058 | 0.0484 | 0.0143 | 0.0042 | 0.0094
MAE por clases: 0.0823 | 0.1329 | 0.0549 | 0.111 | 0.3144 | 0.0691 | 0.0514 | 0.0597 | 0.0932 | 0.0313 - solo en imágenes en las que aparecen
Tiempo de ejecución: 2.55s


,n. noltei,z. marina,g. vermiculophylla,sedimento,arena,roca,algas verdes,algas pardas,algas rojas,microfitobentos
n. noltei,0.00,0.09,0.05,0.33,0.05,0.03,0.35,0.05,0.02,0.04
z. marina,0.66,0.00,0.01,0.16,0.01,0.01,0.09,0.02,0.01,0.03
g. vermiculophylla,0.41,0.36,0.00,0.00,0.01,0.04,0.00,0.00,0.10,0.07
sedimento,0.50,0.05,0.03,0.00,0.06,0.03,0.24,0.07,0.02,0.01
arena,0.34,0.01,0.02,0.43,0.00,0.01,0.13,0.02,0.01,0.03
roca,0.05,0.18,0.10,0.19,0.07,0.00,0.00,0.08,0.11,0.23
algas verdes,0.66,0.03,0.02,0.13,0.02,0.04,0.00,0.03,0.03,0.04
algas pardas,0.10,0.02,0.01,0.58,0.02,0.02,0.23,0.00,0.02,0.01
algas rojas,0.40,0.06,0.02,0.33,0.00,0.00,0.15,0.01,0.00,0.02
microfitobentos,0.41,0.02,0.02,0.27,0.03,0.02,0.18,0.04,0.01,0.00



Modelo de red multiregresión MRConvolutional_Histogram_ConvNeXt_tiny.
Divergencia KL: 0.3440 , MAE Loss: 0.0405
MAE por clases: 0.1037 | 0.0247 | 0.0063 | 0.1062 | 0.0742 | 0.0067 | 0.0516 | 0.0164 | 0.0049 | 0.0103
MAE por clases: 0.0956 | 0.1307 | 0.0394 | 0.1055 | 0.338 | 0.0646 | 0.0618 | 0.0521 | 0.0876 | 0.0313 - solo en imágenes en las que aparecen
Tiempo de ejecución: 2.34s


,n. noltei,z. marina,g. vermiculophylla,sedimento,arena,roca,algas verdes,algas pardas,algas rojas,microfitobentos
n. noltei,0.00,0.08,0.04,0.34,0.08,0.03,0.30,0.07,0.03,0.03
z. marina,0.65,0.00,0.02,0.15,0.03,0.02,0.06,0.03,0.01,0.04
g. vermiculophylla,0.46,0.29,0.00,0.00,0.03,0.07,0.00,0.00,0.06,0.08
sedimento,0.49,0.02,0.03,0.00,0.12,0.04,0.18,0.09,0.02,0.01
arena,0.32,0.01,0.02,0.49,0.00,0.01,0.08,0.03,0.01,0.04
roca,0.03,0.04,0.03,0.54,0.07,0.00,0.00,0.17,0.02,0.10
algas verdes,0.56,0.09,0.02,0.07,0.08,0.04,0.00,0.07,0.03,0.04
algas pardas,0.13,0.01,0.02,0.43,0.03,0.04,0.32,0.00,0.02,0.01
algas rojas,0.43,0.05,0.02,0.35,0.00,0.01,0.10,0.02,0.00,0.02
microfitobentos,0.41,0.02,0.02,0.23,0.06,0.03,0.15,0.07,0.01,0.00
